Import and Seed

In [1]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from torchvision.datasets import MNIST

SEED = 42

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


Gaussian Noise Transform

In [2]:
class GaussianNoise:
    def __init__(self, std=0.0):
        self.std = std

    def __call__(self, tensor):
        if self.std == 0.0:
            return tensor
        noise = torch.randn_like(tensor) * self.std
        return tensor + noise

    def __repr__(self):
        return f"GaussianNoise(std={self.std})"

Model Builder

In [3]:
def build_alexnet(n_classes):
    model = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, n_classes)
    return model.to(device)

Training & Evaluation Helpers

In [4]:
def train_one_epoch(model, loader, optimizer, criterion, dataset_size):
    model.train()
    running_loss = 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0) / dataset_size
    return running_loss


def evaluate(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return correct / total


def train_and_evaluate(model, train_loader, test_loader,
                       n_epochs, dataset_size, label=""):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    print(f"\n")
    print(f"  {label}")
    print(f"\n")
    for epoch in range(1, n_epochs + 1):
        epoch_loss = train_one_epoch(model, train_loader, optimizer,
                                     criterion, dataset_size)
        print(f"  Epoch [{epoch:2d}/{n_epochs}]  Normalised Loss: {epoch_loss:.4f}")

    test_acc = evaluate(model, test_loader)
    print(f"\nFinal Test Accuracy: {test_acc*100:.2f}%")
    return test_acc

MNIST

In [5]:

N_EPOCHS   = 5
BATCH_SIZE = 64

def get_mnist_transform(noise_std=0.0):
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
        GaussianNoise(std=noise_std), 
    ])

mnist_results = {}

for noise_pct in [0, 5, 10]:
    noise_std = noise_pct / 100.0
    seed_everything(SEED)

    train_dataset = MNIST(
        root="/kaggle/working/data", train=True,
        download=True, transform=get_mnist_transform(noise_std)
    )
    test_dataset = MNIST(
        root="/kaggle/working/data", train=False,
        download=True, transform=get_mnist_transform(noise_std)
    )

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                              shuffle=True,  num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=2, pin_memory=True)

    model = build_alexnet(n_classes=10)
    label = f"MNIST  |  Noise = {noise_pct}%"

    acc = train_and_evaluate(model, train_loader, test_loader,
                             N_EPOCHS, len(train_dataset), label=label)
    mnist_results[noise_pct] = acc

print("\nMNIST Summary: ")
for pct, acc in mnist_results.items():
    print(f"  Noise {pct:2d}% → Test Accuracy: {acc*100:.2f}%")

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.05MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.05MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.1MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.7MB/s]


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 207MB/s] 




  MNIST  |  Noise = 0%


  Epoch [ 1/5]  Normalised Loss: 0.5878
  Epoch [ 2/5]  Normalised Loss: 0.1213
  Epoch [ 3/5]  Normalised Loss: 0.1143
  Epoch [ 4/5]  Normalised Loss: 0.1031
  Epoch [ 5/5]  Normalised Loss: 0.1055

Final Test Accuracy: 98.56%


  MNIST  |  Noise = 5%


  Epoch [ 1/5]  Normalised Loss: 0.4674
  Epoch [ 2/5]  Normalised Loss: 0.1210
  Epoch [ 3/5]  Normalised Loss: 0.1036
  Epoch [ 4/5]  Normalised Loss: 0.1014
  Epoch [ 5/5]  Normalised Loss: 0.0987

Final Test Accuracy: 98.84%


  MNIST  |  Noise = 10%


  Epoch [ 1/5]  Normalised Loss: 0.3941
  Epoch [ 2/5]  Normalised Loss: 0.1315
  Epoch [ 3/5]  Normalised Loss: 0.1016
  Epoch [ 4/5]  Normalised Loss: 0.0923
  Epoch [ 5/5]  Normalised Loss: 0.0908

Final Test Accuracy: 98.86%

MNIST Summary: 
  Noise  0% → Test Accuracy: 98.56%
  Noise  5% → Test Accuracy: 98.84%
  Noise 10% → Test Accuracy: 98.86%


Result's Table

In [6]:
print("\n")
print(f"{'Dataset':<25} {'Noise':>6} {'Test Acc':>10}")
print("\n")
for pct, acc in mnist_results.items():
    print(f"{'MNIST':<25} {str(pct)+'%':>6} {acc*100:>9.2f}%")
print("\n")



Dataset                    Noise   Test Acc


MNIST                         0%     98.56%
MNIST                         5%     98.84%
MNIST                        10%     98.86%


